# MS-Dialog Phase 1 — Single-Turn Experiment

**Research question**: When a tech-support specialist model is uncertain about a user's problem, does it ask an *Epistemic* clarifying question (knowledge gap about this specific case) or an *Aleatoric* one (inherent ambiguity in the user's description)?  How does this behaviour vary across models and domains?

**Pipeline per record:**
1. Model sees `category` + `title` + `original_question` → preliminary solution + CQ + confidence
2. User simulator answers the CQ using the synthesised situation summary (hidden from model)
3. Model gives updated solution + updated confidence

**Dataset:** `msdialog_100.jsonl` — 100 MS-Dialog tech-support cases, 19 categories

**CQ classification** is done separately via `run_msdialog_judge.py`

In [1]:
import sys
from pathlib import Path

ROOT       = Path("../../").resolve()
sys.path.insert(0, str(ROOT))

from config import SIMULATOR_MODEL_ID, MSDIALOG_GEMINI_MODEL_ID

DATASET      = "ms-dialog"
DATASETS_DIR = ROOT / "datasets" / DATASET
PROMPTS_DIR  = ROOT / "prompts"  / DATASET
MODEL_ID     = MSDIALOG_GEMINI_MODEL_ID          # specialist / clinician model
OUTPUTS_DIR  = ROOT / "outputs" / DATASET / MODEL_ID

CASES_PATH       = DATASETS_DIR / "msdialog_100.jsonl"
INSTRUCTION_FILE = PROMPTS_DIR  / "phase1_instruction.txt"
OUTPUT_CSV       = OUTPUTS_DIR  / "phase1_singleturn_results.csv"

REQUEST_INTERVAL = 1.0

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Specialist : {MODEL_ID}")
print(f"Simulator  : {SIMULATOR_MODEL_ID}")
print(f"Cases      : {CASES_PATH}")
print(f"Output CSV : {OUTPUT_CSV}")

Specialist : gemini-2.5-flash
Simulator  : gemini-3.1-pro-preview
Cases      : D:\final_project\pilot_study\datasets\ms-dialog\msdialog_100.jsonl
Output CSV : D:\final_project\pilot_study\outputs\ms-dialog\gemini-2.5-flash\phase1_singleturn_results.csv


In [2]:
import importlib
import json
import logging

import src.utils as utils_mod
import src.providers as providers_mod
import src.pipeline as pipeline_mod
importlib.reload(utils_mod)
importlib.reload(providers_mod)
importlib.reload(pipeline_mod)

from src.utils import load_dotenv
from src.providers import GeminiProvider
from src.pipeline import MsDialogPhase1Pipeline, UserSimulator, MSDIALOG_TURN_0_SCHEMA

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(ROOT / ".env")
print("Environment loaded.")

Environment loaded.


## Smoke Test

In [3]:
provider = GeminiProvider(model_id=MODEL_ID)
resp = provider.call(
    system_instruction="You are a helpful assistant.",
    user_message="Reply with exactly: SMOKE TEST PASSED",
    temperature=0.0, max_tokens=5000,
)
assert resp and "SMOKE" in resp.upper(), f"Smoke test failed: {resp!r}"
print(f"Specialist smoke test PASSED: {resp.strip()}")

sim_provider = GeminiProvider(model_id=SIMULATOR_MODEL_ID)
resp2 = sim_provider.call(
    system_instruction="You are a helpful assistant.",
    user_message="Reply with exactly: SMOKE TEST PASSED",
    temperature=0.0, max_tokens=5000,
)
assert resp2 and "SMOKE" in resp2.upper(), f"Simulator smoke test failed: {resp2!r}"
print(f"Simulator  smoke test PASSED: {resp2.strip()}")

12:41:32 [INFO] src.providers — GeminiProvider ready — model=gemini-2.5-flash api_version=v1beta


12:41:32 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


12:41:33 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


12:41:33 [INFO] src.providers — GeminiProvider ready — model=gemini-3.1-pro-preview api_version=v1beta


12:41:33 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


Specialist smoke test PASSED: SMOKE TEST PASSED


12:41:36 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


Simulator  smoke test PASSED: SMOKE TEST PASSED


## Load Dataset

In [4]:
with open(CASES_PATH, encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Loaded {len(records)} records")
print(f"Fields: {list(records[0].keys())}")

from collections import Counter
cats = Counter(r["category"] for r in records)
print(f"\nCategories ({len(cats)} unique):")
for cat, n in cats.most_common():
    print(f"  {cat:35s}: {n}")

Loaded 100 records
Fields: ['case_id', 'title', 'category', 'original_question', 'simulator_context', 'raw_simulator_context', 'accepted_answer', 'source_dialog_id', 'answer_source', 'context_richness', 'n_utterances', 'has_fd', 'has_agent_cq']

Categories (19 unique):
  PowerPoint                         : 17
  Excel                              : 14
  Apps_Windows_10                    : 8
  Word                               : 7
  Bing_Apps                          : 6
  Bing                               : 6
  Windows_8.1                        : 5
  Bing_Search                        : 5
  Bing_Maps                          : 5
  Skype_iOS                          : 5
  Windows_7                          : 4
  Windows_10                         : 4
  Skype_Android                      : 3
  Windows_RT_8.1                     : 3
  Skype_Windows_Desktop              : 2
  Skype_Windows_10                   : 2
  Outlook_Email                      : 2
  Games_Windows_10             

## Dry Run — Single Record
Verify the two-turn flow on one record before running everything.

In [5]:
from src.pipeline import _format_problem, _MSDIALOG_FINAL_INSTRUCTION, MSDIALOG_FINAL_SCHEMA
from src.utils import parse_json_response
import json as _json

simulator = UserSimulator(sim_provider)
instruction = INSTRUCTION_FILE.read_text(encoding="utf-8").strip()

test = records[0]
test_id = test.get("id") or test.get("case_id")
print(f"Testing on: {test_id} | {test['category']}")
print(f"Title: {test['title']}")
print(f"OQ: {test['original_question'][:200]}")
print()

# Turn 0
raw_0 = provider.call(
    system_instruction=instruction,
    user_message=_format_problem(test["title"], test["category"], test["original_question"]),
    temperature=0.0, max_tokens=3000, expect_json=MSDIALOG_TURN_0_SCHEMA,
)
p0 = parse_json_response(raw_0)
print(f"=== TURN 0 ===\n{_json.dumps(p0, indent=2)}")

# Simulate user response
sim_resp = simulator.answer(p0["clarifying_question"], test["simulator_context"])
print(f"\n=== USER RESPONSE ===\n{sim_resp}")

# Turn 1 — final solution
user_msg_1 = (
    f"{_format_problem(test['title'], test['category'], test['original_question'])}\n\n"
    f"Your clarifying question:\n{p0['clarifying_question']}\n\n"
    f"User's answer:\n{sim_resp}\n\n"
    f"Based on this additional information, provide your updated solution."
)
raw_1 = provider.call(
    system_instruction=_MSDIALOG_FINAL_INSTRUCTION,
    user_message=user_msg_1,
    temperature=0.0, max_tokens=3000, expect_json=MSDIALOG_FINAL_SCHEMA,
)
p1 = parse_json_response(raw_1)
print(f"\n=== TURN 1 (FINAL) ===\n{_json.dumps(p1, indent=2)}")
print(f"\n--- ACCEPTED ANSWER ---\n{test['accepted_answer'][:300]}")

12:41:36 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


Testing on: msd_001 | Windows_8.1
Title: icon deleted from task bar
OQ: I accidently deleted an icon for microsoft solitaire from my task bar.   Is there a way to get it back?



12:41:39 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


12:41:39 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


=== TURN 0 ===
{
  "clarifying_question": "Can you still find and launch Microsoft Solitaire from your Start Menu or Apps list?",
  "preliminary_solution": "To restore the icon, first search for 'Microsoft Solitaire Collection' in your Start Menu. Once you find it, right-click on the application icon. From the context menu, select 'Pin to taskbar'. This should place the icon back on your taskbar.",
  "confidence": 95
}


12:41:45 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"


12:41:45 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.



=== USER RESPONSE ===
I haven't checked. I just noticed the icon disappeared from the bottom of my screen.


12:41:49 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"



=== TURN 1 (FINAL) ===
{
  "final_solution": "1. Open the Start Menu and search for \"Microsoft Solitaire\".2. Once found, right-click on the Microsoft Solitaire application.3. Select \"Pin to taskbar\" from the context menu. This will restore the icon to your taskbar.",
  "confidence": 100
}

--- ACCEPTED ANSWER ---
Hi,  This covers how to get a Desktop Shortcut and get your Taskbar icon back:     Left click on far left Icon in the Taskbar > then in the "Start" Window click on down Arrow at bottom left to go to "Apps by Category" Window > locate the App > right click on it and select "Open file location" > in t


## Run Full Experiment
100 cases × 1 CQ round. Resumes automatically if interrupted.

In [6]:
pipeline = MsDialogPhase1Pipeline(
    provider=provider,
    instruction_file=INSTRUCTION_FILE,
    output_csv=OUTPUT_CSV,
    request_interval=REQUEST_INTERVAL,
    simulator_provider=sim_provider,
)

pipeline.run(records)

12:41:49 [INFO] src.pipeline — MsDialogPhase1Pipeline ready — specialist=gemini/gemini-2.5-flash simulator=gemini/gemini-3.1-pro-preview output=D:\final_project\pilot_study\outputs\ms-dialog\gemini-2.5-flash\phase1_singleturn_results.csv


12:41:49 [INFO] src.pipeline — [1/100] Processing msd_001 (Windows_8.1)


12:41:49 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


12:41:52 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


12:41:52 [INFO] src.pipeline —   CQ: Can you still find Microsoft Solitaire in your Start Menu or by searching for it on your computer?


...

13:40:50 [INFO] src.pipeline —   User: I'm not sure about that.


13:40:51 [INFO] google_genai.models — AFC is enabled with max remote calls: 10.


13:40:58 [INFO] httpx — HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent "HTTP/1.1 200 OK"


13:40:58 [INFO] src.pipeline —   Updated conf=90


13:40:59 [INFO] src.pipeline — MsDialog Phase1 complete — total=100 succeeded=100 skipped=0 failed=0


## Results Summary

In [7]:
import pandas as pd
from IPython.display import display

df = pd.read_csv(OUTPUT_CSV)
valid = df[~df["was_blocked"]]

print(f"Total: {len(df)} | Blocked: {df['was_blocked'].sum()} | Valid: {len(valid)}")
print(f"\nMean preliminary confidence : {valid['preliminary_confidence'].mean():.1f}")
print(f"Mean updated confidence     : {valid['updated_confidence'].mean():.1f}")
print(f"Mean confidence delta       : {(valid['updated_confidence'] - valid['preliminary_confidence']).mean():.1f}")

print("\nSample CQs generated:")
id_col = "id" if "id" in valid.columns else "case_id"
for _, row in valid.head(5).iterrows():
    print(f"  [{row[id_col]}] {row['clarifying_question'][:100]}")

print("\nNote: CQ classification (EPISTEMIC/ALEATORIC) → run run_msdialog_judge.py")

Total: 100 | Blocked: 0 | Valid: 100

Mean preliminary confidence : 79.7
Mean updated confidence     : 90.5
Mean confidence delta       : 10.9

Sample CQs generated:
  [msd_001] Can you still find Microsoft Solitaire in your Start Menu or by searching for it on your computer?
  [msd_002] Are you referring to the desktop calculator application or the full-screen app from the Start screen
  [msd_003] Have you tried resetting the Bing News app data or reinstalling the app from the Windows Store since
  [msd_004] Has there been any recent software installation, update (Windows or Office), or new Excel add-in ena
  [msd_005] Have you tried creating a new user account and attempting to launch 'Back up your computer' from the

Note: CQ classification (EPISTEMIC/ALEATORIC) → run run_msdialog_judge.py
